In [4]:
! pip install transformers==4.35.2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizerBase


PastKeyValues = Tuple[Tuple[torch.Tensor, torch.Tensor], ...]


@dataclass
class TrainConfig:
    model_a_id: str = "gpt2"
    model_b_id: str = "gpt2-medium"
    output_dir: str = "./outputs/lsc_toy"
    max_steps: int = 500
    batch_size: int = 1
    grad_accum_steps: int = 16
    total_tokens: int = 128
    prefix_tokens: int = 64
    learning_rate: float = 1e-4
    weight_decay: float = 1e-2
    warmup_steps: int = 50
    grad_clip_norm: float = 1.0
    log_every: int = 25
    save_every: int = 100
    seed: int = 42
    shuffle_buffer: int = 50_000
    top_layers_to_translate: int = 6
    translator_dim: int = 1024
    translator_heads: int = 16
    translator_depth: int = 2
    translator_mlp_ratio: int = 2
    train_directions: str = "B_to_A"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: str = "float32"


@dataclass
class ModelSpec:
    model_id: str
    num_layers: int
    hidden_size: int
    num_heads: int
    head_dim: int


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_torch_dtype(dtype_name: str) -> torch.dtype:
    mapping = {
        "float32": torch.float32,
        "float": torch.float32,
        "fp32": torch.float32,
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
    }
    key = dtype_name.lower()
    if key not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_name}")
    return mapping[key]


def parse_train_directions(train_directions: str) -> List[str]:
    allowed = {"A_to_B", "B_to_A"}
    parsed = [item.strip() for item in train_directions.split(",") if item.strip()]
    if not parsed:
        raise ValueError("train_directions must contain at least one direction.")

    deduped = []
    seen = set()
    for item in parsed:
        if item not in allowed:
            raise ValueError(
                f"Unsupported train direction: {item}. "
                f"Allowed values are: {sorted(allowed)}"
            )
        if item not in seen:
            deduped.append(item)
            seen.add(item)
    return deduped


def load_tokenizer(model_id: str) -> PreTrainedTokenizerBase:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def freeze_model(model: PreTrainedModel) -> None:
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)


def load_frozen_model(model_id: str, device: str, dtype: str = "float32") -> PreTrainedModel:
    torch_dtype = get_torch_dtype(dtype)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch_dtype)
    model.to(device)
    freeze_model(model)
    return model


def get_model_spec(model: PreTrainedModel) -> ModelSpec:
    config = model.config
    num_heads = getattr(config, "n_head", None)
    hidden_size = getattr(config, "n_embd", None)
    num_layers = getattr(config, "n_layer", None)
    if num_heads is None or hidden_size is None or num_layers is None:
        raise ValueError("This toy example expects GPT-2 style configs with n_head/n_embd/n_layer.")
    if hidden_size % num_heads != 0:
        raise ValueError("hidden_size must be divisible by num_heads.")
    return ModelSpec(
        model_id=getattr(config, "_name_or_path", "unknown"),
        num_layers=num_layers,
        hidden_size=hidden_size,
        num_heads=num_heads,
        head_dim=hidden_size // num_heads,
    )


class OpenWebTextSequenceStream(IterableDataset):
    """
    Streams OpenWebText and yields fixed-length token chunks.

    This behaves like a rolling token buffer, which is usually a better fit for
    language-model style toy experiments than per-document truncation.
    """

    def __init__(
        self,
        tokenizer: PreTrainedTokenizerBase,
        sequence_length: int,
        shuffle_buffer: int = 10_000,
        seed: int = 42,
    ) -> None:
        super().__init__()
        self.tokenizer = tokenizer
        self.sequence_length = sequence_length
        self.shuffle_buffer = shuffle_buffer
        self.seed = seed

    def __iter__(self) -> Iterable[torch.Tensor]:
        stream = load_dataset("openwebtext", split="train", streaming=True)
        stream = stream.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)
        token_buffer: List[int] = []
        eos_token_id = self.tokenizer.eos_token_id
        for example in stream:
            text = example.get("text", "")
            if not text or text.isspace():
                continue
            token_ids = self.tokenizer(text, add_special_tokens=False).input_ids
            if len(token_ids) < 8:
                continue
            token_buffer.extend(token_ids)
            token_buffer.append(eos_token_id)
            while len(token_buffer) >= self.sequence_length:
                chunk = token_buffer[: self.sequence_length]
                token_buffer = token_buffer[self.sequence_length :]
                yield torch.tensor(chunk, dtype=torch.long)


class SelfAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: int = 2) -> None:
        super().__init__()
        self.attn_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        norm_hidden = self.attn_norm(hidden)
        attn_out, _ = self.attn(norm_hidden, norm_hidden, norm_hidden, need_weights=False)
        hidden = hidden + attn_out
        hidden = hidden + self.ffn(self.ffn_norm(hidden))
        return hidden


class PerLayerTranslator(nn.Module):
    def __init__(
        self,
        src_hidden_size: int,
        dst_hidden_size: int,
        translator_dim: int,
        translator_heads: int,
        translator_depth: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.input_norm = nn.LayerNorm(src_hidden_size)
        self.input_proj = nn.Linear(src_hidden_size, translator_dim)
        self.blocks = nn.ModuleList(
            [
                SelfAttentionBlock(
                    dim=translator_dim,
                    num_heads=translator_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(translator_depth)
            ]
        )
        self.output_norm = nn.LayerNorm(translator_dim)
        self.output_proj = nn.Linear(translator_dim, dst_hidden_size)

    def forward(self, layer_cache: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.input_proj(self.input_norm(layer_cache)))
        for block in self.blocks:
            hidden = block(hidden)
        return self.output_proj(self.output_norm(hidden))


class TopLayerDirectionalTranslator(nn.Module):
    def __init__(
        self,
        src_hidden_size: int,
        dst_hidden_size: int,
        top_layers_to_translate: int,
        translator_dim: int,
        translator_heads: int,
        translator_depth: int,
        mlp_ratio: int,
    ) -> None:
        super().__init__()
        self.top_layers_to_translate = top_layers_to_translate
        self.key_layers = nn.ModuleList(
            [
                PerLayerTranslator(
                    src_hidden_size=src_hidden_size,
                    dst_hidden_size=dst_hidden_size,
                    translator_dim=translator_dim,
                    translator_heads=translator_heads,
                    translator_depth=translator_depth,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(top_layers_to_translate)
            ]
        )
        self.value_layers = nn.ModuleList(
            [
                PerLayerTranslator(
                    src_hidden_size=src_hidden_size,
                    dst_hidden_size=dst_hidden_size,
                    translator_dim=translator_dim,
                    translator_heads=translator_heads,
                    translator_depth=translator_depth,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(top_layers_to_translate)
            ]
        )

    def forward(self, key_block: torch.Tensor, value_block: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        key_outputs = []
        value_outputs = []
        for layer_idx in range(self.top_layers_to_translate):
            key_outputs.append(self.key_layers[layer_idx](key_block[:, :, layer_idx, :]).unsqueeze(2))
            value_outputs.append(self.value_layers[layer_idx](value_block[:, :, layer_idx, :]).unsqueeze(2))
        translated_key = torch.cat(key_outputs, dim=2)
        translated_value = torch.cat(value_outputs, dim=2)
        return translated_key, translated_value


class TopLayerTranslatorPool(nn.Module):
    def __init__(
        self,
        model_specs: Dict[str, ModelSpec],
        top_layers_to_translate: int,
        translator_dim: int,
        translator_heads: int,
        translator_depth: int,
        mlp_ratio: int,
        active_directions: List[str],
    ) -> None:
        super().__init__()
        spec_a = model_specs["A"]
        spec_b = model_specs["B"]
        max_allowed = min(spec_a.num_layers, spec_b.num_layers)
        if top_layers_to_translate > max_allowed:
            raise ValueError(
                f"top_layers_to_translate={top_layers_to_translate} exceeds min layer count {max_allowed}."
            )
        if not active_directions:
            raise ValueError("active_directions must contain at least one direction.")

        self.model_specs = model_specs
        self.top_layers_to_translate = top_layers_to_translate
        self.active_directions = tuple(active_directions)

        adapters = {}
        if "A_to_B" in self.active_directions:
            adapters["A_to_B"] = TopLayerDirectionalTranslator(
                src_hidden_size=spec_a.hidden_size,
                dst_hidden_size=spec_b.hidden_size,
                top_layers_to_translate=top_layers_to_translate,
                translator_dim=translator_dim,
                translator_heads=translator_heads,
                translator_depth=translator_depth,
                mlp_ratio=mlp_ratio,
            )
        if "B_to_A" in self.active_directions:
            adapters["B_to_A"] = TopLayerDirectionalTranslator(
                src_hidden_size=spec_b.hidden_size,
                dst_hidden_size=spec_a.hidden_size,
                top_layers_to_translate=top_layers_to_translate,
                translator_dim=translator_dim,
                translator_heads=translator_heads,
                translator_depth=translator_depth,
                mlp_ratio=mlp_ratio,
            )

        self.adapters = nn.ModuleDict(adapters)

    def translate_top_layer_blocks(
        self,
        key_block: torch.Tensor,
        value_block: torch.Tensor,
        src_name: str,
        dst_name: str,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        adapter_name = f"{src_name}_to_{dst_name}"
        if adapter_name not in self.adapters:
            raise ValueError(
                f"Translator direction {adapter_name} is not available. "
                f"Active directions: {list(self.active_directions)}"
            )
        return self.adapters[adapter_name](key_block, value_block)

    def translate_top_layers(
        self,
        past_key_values: PastKeyValues,
        src_name: str,
        dst_name: str,
        dst_spec: ModelSpec,
    ) -> PastKeyValues:
        key_block, value_block = extract_top_layer_blocks(
            past_key_values=past_key_values,
            top_layers_to_translate=self.top_layers_to_translate,
        )
        translated_key, translated_value = self.translate_top_layer_blocks(
            key_block=key_block,
            value_block=value_block,
            src_name=src_name,
            dst_name=dst_name,
        )
        return blocks_to_partial_past_key_values(
            key_block=translated_key,
            value_block=translated_value,
            num_heads=dst_spec.num_heads,
            head_dim=dst_spec.head_dim,
        )


@torch.no_grad()
def extract_past_key_values(model: PreTrainedModel, input_ids: torch.Tensor) -> PastKeyValues:
    outputs = model(input_ids=input_ids, use_cache=True)
    return outputs.past_key_values


def past_key_values_to_blocks(past_key_values: PastKeyValues) -> Tuple[torch.Tensor, torch.Tensor]:
    key_layers = []
    value_layers = []
    for key, value in past_key_values:
        batch_size, num_heads, seq_len, head_dim = key.shape
        key_flat = key.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        value_flat = value.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        key_layers.append(key_flat)
        value_layers.append(value_flat)
    key_block = torch.stack(key_layers, dim=2)
    value_block = torch.stack(value_layers, dim=2)
    return key_block, value_block


def extract_top_layer_blocks(
    past_key_values: PastKeyValues,
    top_layers_to_translate: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    if top_layers_to_translate < 1:
        raise ValueError("top_layers_to_translate must be >= 1")
    if top_layers_to_translate > len(past_key_values):
        raise ValueError(
            f"Cannot extract {top_layers_to_translate} layers from cache with only {len(past_key_values)} layers."
        )
    return past_key_values_to_blocks(past_key_values[-top_layers_to_translate:])


def blocks_to_partial_past_key_values(
    key_block: torch.Tensor,
    value_block: torch.Tensor,
    num_heads: int,
    head_dim: int,
) -> PastKeyValues:
    batch_size, seq_len, num_layers, hidden_size = key_block.shape
    expected_hidden = num_heads * head_dim
    if hidden_size != expected_hidden:
        raise ValueError(f"Hidden mismatch: block has {hidden_size}, expected {expected_hidden}.")

    past_key_values = []
    for layer_idx in range(num_layers):
        key_layer = key_block[:, :, layer_idx, :]
        value_layer = value_block[:, :, layer_idx, :]
        key_layer = key_layer.view(batch_size, seq_len, num_heads, head_dim)
        value_layer = value_layer.view(batch_size, seq_len, num_heads, head_dim)
        key_layer = key_layer.permute(0, 2, 1, 3).contiguous()
        value_layer = value_layer.permute(0, 2, 1, 3).contiguous()
        past_key_values.append((key_layer, value_layer))
    return tuple(past_key_values)


def slice_top_layers(
    past_key_values: PastKeyValues,
    top_layers_to_translate: int,
) -> PastKeyValues:
    if top_layers_to_translate > len(past_key_values):
        raise ValueError(
            f"Cannot slice {top_layers_to_translate} layers from cache with only {len(past_key_values)} layers."
        )
    return tuple(past_key_values[-top_layers_to_translate:])


def replace_top_layers(
    base_past_key_values: PastKeyValues,
    translated_top_past_key_values: PastKeyValues,
) -> PastKeyValues:
    num_replace = len(translated_top_past_key_values)
    if num_replace < 1:
        raise ValueError("translated_top_past_key_values must contain at least one layer.")
    if num_replace > len(base_past_key_values):
        raise ValueError(
            f"Cannot replace {num_replace} layers in cache with only {len(base_past_key_values)} layers."
        )

    base_list = list(base_past_key_values)
    start_idx = len(base_list) - num_replace

    for offset, translated_layer in enumerate(translated_top_past_key_values):
        base_key, base_value = base_list[start_idx + offset]
        translated_key, translated_value = translated_layer

        if base_key.shape != translated_key.shape:
            raise ValueError(
                f"Key shape mismatch at replaced layer {offset}: "
                f"base={tuple(base_key.shape)} vs translated={tuple(translated_key.shape)}"
            )
        if base_value.shape != translated_value.shape:
            raise ValueError(
                f"Value shape mismatch at replaced layer {offset}: "
                f"base={tuple(base_value.shape)} vs translated={tuple(translated_value.shape)}"
            )

        base_list[start_idx + offset] = (translated_key, translated_value)

    return tuple(base_list)


def flatten_past_key_values(past_key_values: PastKeyValues) -> torch.Tensor:
    flat_parts = []
    for key, value in past_key_values:
        flat_parts.append(key.reshape(key.shape[0], -1))
        flat_parts.append(value.reshape(value.shape[0], -1))
    return torch.cat(flat_parts, dim=1)


def cosine_similarity_between_past(a: PastKeyValues, b: PastKeyValues) -> float:
    flat_a = flatten_past_key_values(a)
    flat_b = flatten_past_key_values(b)
    return F.cosine_similarity(flat_a, flat_b, dim=1).mean().item()


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(param.numel() for param in module.parameters() if param.requires_grad)


def move_past_to_device(past_key_values: PastKeyValues, device: str) -> PastKeyValues:
    moved = []
    for key, value in past_key_values:
        moved.append((key.to(device), value.to(device)))
    return tuple(moved)


def compute_suffix_lm_loss(
    target_model: PreTrainedModel,
    past_key_values: PastKeyValues,
    lm_input_ids: torch.Tensor,
    lm_labels: torch.Tensor,
) -> torch.Tensor:
    outputs = target_model(
        input_ids=lm_input_ids,
        past_key_values=past_key_values,
        use_cache=False,
    )
    logits = outputs.logits
    vocab_size = logits.shape[-1]
    return F.cross_entropy(
        logits.reshape(-1, vocab_size),
        lm_labels.reshape(-1),
        reduction="mean",
    )


class InfiniteDataLoader:
    def __init__(self, dataloader: DataLoader) -> None:
        self.dataloader = dataloader
        self.iterator = iter(self.dataloader)

    def __iter__(self):
        return self

    def __next__(self):
        try:
            return next(self.iterator)
        except StopIteration:
            self.iterator = iter(self.dataloader)
            return next(self.iterator)


def build_training_dataloader(tokenizer: PreTrainedTokenizerBase, config: TrainConfig) -> InfiniteDataLoader:
    dataset = OpenWebTextSequenceStream(
        tokenizer=tokenizer,
        sequence_length=config.total_tokens,
        shuffle_buffer=config.shuffle_buffer,
        seed=config.seed,
    )
    dataloader = DataLoader(dataset, batch_size=config.batch_size, num_workers=0)
    return InfiniteDataLoader(dataloader)


def split_prefix_and_suffix_for_exact_next_token_loss(
    input_ids: torch.Tensor,
    prefix_tokens: int,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if prefix_tokens < 2:
        raise ValueError("prefix_tokens must be >= 2")
    prefix_cache_ids = input_ids[:, : prefix_tokens - 1]
    lm_input_ids = input_ids[:, prefix_tokens - 1 : -1]
    lm_labels = input_ids[:, prefix_tokens:]
    return prefix_cache_ids, lm_input_ids, lm_labels


class WarmupCosineScheduler:
    def __init__(self, optimizer: torch.optim.Optimizer, warmup_steps: int, total_steps: int) -> None:
        self.optimizer = optimizer
        self.warmup_steps = max(1, warmup_steps)
        self.total_steps = max(self.warmup_steps + 1, total_steps)
        self.step_id = 0
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]

    def step(self) -> None:
        self.step_id += 1
        if self.step_id <= self.warmup_steps:
            multiplier = self.step_id / self.warmup_steps
        else:
            progress = (self.step_id - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            multiplier = 0.5 * (1.0 + math.cos(math.pi * progress))
        for base_lr, param_group in zip(self.base_lrs, self.optimizer.param_groups):
            param_group["lr"] = base_lr * multiplier

    @property
    def lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]


def build_translator_pool(
    model_a: PreTrainedModel,
    model_b: PreTrainedModel,
    config: TrainConfig,
) -> Tuple[TopLayerTranslatorPool, Dict[str, ModelSpec]]:
    model_specs = {
        "A": get_model_spec(model_a),
        "B": get_model_spec(model_b),
    }
    active_directions = parse_train_directions(config.train_directions)
    translator_pool = TopLayerTranslatorPool(
        model_specs=model_specs,
        top_layers_to_translate=config.top_layers_to_translate,
        translator_dim=config.translator_dim,
        translator_heads=config.translator_heads,
        translator_depth=config.translator_depth,
        mlp_ratio=config.translator_mlp_ratio,
        active_directions=active_directions,
    )
    translator_pool.to(config.device)
    return translator_pool, model_specs


def build_models_and_tokenizer(config: TrainConfig) -> Tuple[PreTrainedModel, PreTrainedModel, PreTrainedTokenizerBase]:
    tokenizer = load_tokenizer(config.model_a_id)
    model_a = load_frozen_model(config.model_a_id, device=config.device, dtype=config.dtype)
    model_b = load_frozen_model(config.model_b_id, device=config.device, dtype=config.dtype)
    return model_a, model_b, tokenizer


def save_checkpoint(
    output_path: str,
    translator_pool: TopLayerTranslatorPool,
    optimizer: torch.optim.Optimizer,
    scheduler: WarmupCosineScheduler,
    train_config: TrainConfig,
    step: int,
    extra: Optional[Dict] = None,
) -> None:
    payload = {
        "translator_pool": translator_pool.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
        "train_config": asdict(train_config),
        "scheduler_step": scheduler.step_id,
    }
    if extra is not None:
        payload["extra"] = extra
    output_path = str(output_path)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, output_path)


def load_train_config_from_checkpoint(checkpoint_path: str) -> TrainConfig:
    payload = torch.load(checkpoint_path, map_location="cpu")
    return TrainConfig(**payload["train_config"])


def load_translator_pool_from_checkpoint(
    checkpoint_path: str,
    device_override: Optional[str] = None,
) -> Tuple[
    TrainConfig,
    TopLayerTranslatorPool,
    Dict[str, ModelSpec],
    PreTrainedModel,
    PreTrainedModel,
    PreTrainedTokenizerBase,
]:
    payload = torch.load(checkpoint_path, map_location="cpu")
    config = TrainConfig(**payload["train_config"])
    if device_override is not None:
        config.device = device_override
    model_a, model_b, tokenizer = build_models_and_tokenizer(config)
    translator_pool, model_specs = build_translator_pool(model_a, model_b, config)
    translator_pool.load_state_dict(payload["translator_pool"])
    translator_pool.to(config.device)
    translator_pool.eval()
    return config, translator_pool, model_specs, model_a, model_b, tokenizer


def write_json(path: str, payload: Dict) -> None:
    path_obj = Path(path)
    path_obj.parent.mkdir(parents=True, exist_ok=True)
    with path_obj.open("w", encoding="utf-8") as fp:
        json.dump(payload, fp, indent=2, ensure_ascii=False)

In [6]:
"""
Code 1
- Goal: train a toy cross-attention translator between GPT-2 and GPT-2 Medium
- Data: OpenWebText
- Objective: suffix LM loss only (bidirectional sum per step), no reconstruction loss

Important
- This example is intentionally compact and pragmatic.
- It is written to be compatible with transformers==4.35.2, where past_key_values are plain tuples,
  so DynamicCache is not used.
"""

import logging
from pathlib import Path

import torch
from tqdm.auto import tqdm


CONFIG = TrainConfig(
    model_a_id="gpt2",
    model_b_id="gpt2-medium",
    output_dir="./outputs/lsc_toy",
    max_steps=500,
    batch_size=1,
    grad_accum_steps=16,
    total_tokens=128,
    prefix_tokens=64,
    learning_rate=1e-4,
    weight_decay=1e-2,
    warmup_steps=50,
    grad_clip_norm=1.0,
    log_every=25,
    save_every=100,
    seed=42,
    shuffle_buffer=50_000,
    top_layers_to_translate=6,
    translator_dim=1024,
    translator_heads=16,
    translator_depth=2,
    translator_mlp_ratio=2,
    train_directions="B_to_A",
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype="float32",
)


def setup_logger(log_path: Path) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("lsc_train")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger


def main() -> None:
    set_seed(CONFIG.seed)
    output_dir = Path(CONFIG.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    write_json(output_dir / "train_config.json", CONFIG.__dict__)

    log_path = output_dir / "train.log"
    logger = setup_logger(log_path)
    logger.info("Starting training")
    logger.info("train_config=%s", CONFIG.__dict__)

    train_directions = parse_train_directions(CONFIG.train_directions)
    logger.info("train_directions=%s", train_directions)

    logger.info("[Setup] device=%s", CONFIG.device)
    logger.info("[Setup] loading models: A=%s, B=%s", CONFIG.model_a_id, CONFIG.model_b_id)
    model_a, model_b, tokenizer = build_models_and_tokenizer(CONFIG)
    translator_pool, model_specs = build_translator_pool(model_a, model_b, CONFIG)
    translator_pool.train()

    logger.info("[Setup] full model specs")
    logger.info(
        "  A: layers=%d, hidden=%d, heads=%d",
        model_specs["A"].num_layers,
        model_specs["A"].hidden_size,
        model_specs["A"].num_heads,
    )
    logger.info(
        "  B: layers=%d, hidden=%d, heads=%d",
        model_specs["B"].num_layers,
        model_specs["B"].hidden_size,
        model_specs["B"].num_heads,
    )
    logger.info("[Setup] top_layers_to_translate = %d", CONFIG.top_layers_to_translate)
    logger.info("[Setup] trainable translator params = %s", f"{count_trainable_parameters(translator_pool):,}")

    dataloader = build_training_dataloader(tokenizer, CONFIG)

    optimizer = torch.optim.AdamW(
        translator_pool.parameters(),
        lr=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
    )
    scheduler = WarmupCosineScheduler(
        optimizer=optimizer,
        warmup_steps=CONFIG.warmup_steps,
        total_steps=CONFIG.max_steps,
    )

    running_loss = 0.0
    progress_bar = tqdm(range(1, CONFIG.max_steps + 1), desc="Training")

    for step in progress_bar:
        optimizer.zero_grad(set_to_none=True)
        step_loss_value = 0.0

        for _ in range(CONFIG.grad_accum_steps):
            input_ids = next(dataloader).to(CONFIG.device)
            prefix_cache_ids, lm_input_ids, lm_labels = split_prefix_and_suffix_for_exact_next_token_loss(
                input_ids=input_ids,
                prefix_tokens=CONFIG.prefix_tokens,
            )

            with torch.no_grad():
                past_a = extract_past_key_values(model_a, prefix_cache_ids)
                past_b = extract_past_key_values(model_b, prefix_cache_ids)

            direction_contexts = {
                "A_to_B": {
                    "source_past": past_a,
                    "source_name": "A",
                    "target_name": "B",
                    "target_spec": model_specs["B"],
                    "target_full_past": past_b,
                    "target_model": model_b,
                },
                "B_to_A": {
                    "source_past": past_b,
                    "source_name": "B",
                    "target_name": "A",
                    "target_spec": model_specs["A"],
                    "target_full_past": past_a,
                    "target_model": model_a,
                },
            }

            total_direction_loss = 0.0
            for direction in train_directions:
                context = direction_contexts[direction]

                translated_top_past = translator_pool.translate_top_layers(
                    past_key_values=context["source_past"],
                    src_name=context["source_name"],
                    dst_name=context["target_name"],
                    dst_spec=context["target_spec"],
                )
                mixed_target_past = replace_top_layers(
                    base_past_key_values=context["target_full_past"],
                    translated_top_past_key_values=translated_top_past,
                )
                direction_loss = compute_suffix_lm_loss(
                    target_model=context["target_model"],
                    past_key_values=mixed_target_past,
                    lm_input_ids=lm_input_ids,
                    lm_labels=lm_labels,
                )
                total_direction_loss = total_direction_loss + direction_loss

            loss = total_direction_loss / CONFIG.grad_accum_steps
            loss.backward()
            step_loss_value += loss.item()

        torch.nn.utils.clip_grad_norm_(translator_pool.parameters(), CONFIG.grad_clip_norm)
        optimizer.step()
        scheduler.step()

        running_loss += step_loss_value
        if step % CONFIG.log_every == 0:
            avg_loss = running_loss / CONFIG.log_every
            progress_bar.set_postfix(
                loss=f"{avg_loss:.4f}",
                lr=f"{scheduler.lr:.2e}",
            )
            logger.info(
                "[Step %04d] total_suffix_lm_loss=%.4f | lr=%.2e | train_directions=%s",
                step,
                avg_loss,
                scheduler.lr,
                ",".join(train_directions),
            )
            running_loss = 0.0

        # if step % CONFIG.save_every == 0:
        #     checkpoint_path = output_dir / f"checkpoint_step_{step:04d}.pt"
        #     save_checkpoint(
        #         output_path=str(checkpoint_path),
        #         translator_pool=translator_pool,
        #         optimizer=optimizer,
        #         scheduler=scheduler,
        #         train_config=CONFIG,
        #         step=step,
        #         extra={
        #             "note": "Toy checkpoint trained with suffix LM loss only.",
        #             "model_a": CONFIG.model_a_id,
        #             "model_b": CONFIG.model_b_id,
        #             "top_layers_to_translate": CONFIG.top_layers_to_translate,
        #             "train_directions": CONFIG.train_directions,
        #         },
        #     )
        #     logger.info("[Checkpoint] saved to %s", checkpoint_path)

    final_path = output_dir / "final_checkpoint.pt"
    save_checkpoint(
        output_path=str(final_path),
        translator_pool=translator_pool,
        optimizer=optimizer,
        scheduler=scheduler,
        train_config=CONFIG,
        step=CONFIG.max_steps,
        extra={
            "note": "Final toy checkpoint trained with suffix LM loss only.",
            "model_a": CONFIG.model_a_id,
            "model_b": CONFIG.model_b_id,
            "top_layers_to_translate": CONFIG.top_layers_to_translate,
            "train_directions": CONFIG.train_directions,
        },
    )
    logger.info("[Done] final checkpoint saved to %s", final_path)
    logger.info("Saved train log to %s", log_path)


if __name__ == "__main__":
    main()


2026-03-09 15:07:53,642 | INFO | Starting training
2026-03-09 15:07:53,643 | INFO | train_config={'model_a_id': 'gpt2', 'model_b_id': 'gpt2-medium', 'output_dir': './outputs/lsc_toy', 'max_steps': 500, 'batch_size': 1, 'grad_accum_steps': 16, 'total_tokens': 128, 'prefix_tokens': 64, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'warmup_steps': 50, 'grad_clip_norm': 1.0, 'log_every': 25, 'save_every': 100, 'seed': 42, 'shuffle_buffer': 50000, 'top_layers_to_translate': 6, 'translator_dim': 1024, 'translator_heads': 16, 'translator_depth': 2, 'translator_mlp_ratio': 2, 'train_directions': 'B_to_A', 'device': 'cuda', 'dtype': 'float32'}
2026-03-09 15:07:53,644 | INFO | train_directions=['B_to_A']
2026-03-09 15:07:53,645 | INFO | [Setup] device=cuda
2026-03-09 15:07:53,646 | INFO | [Setup] loading models: A=gpt2, B=gpt2-medium
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. 

Training:   0%|          | 0/500 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1199 > 1024). Running this sequence through the model will result in indexing errors
2026-03-09 15:09:10,990 | INFO | [Step 0025] total_suffix_lm_loss=4.1943 | lr=5.00e-05 | train_directions=B_to_A
2026-03-09 15:10:03,038 | INFO | [Step 0050] total_suffix_lm_loss=3.7380 | lr=1.00e-04 | train_directions=B_to_A
2026-03-09 15:10:54,274 | INFO | [Step 0075] total_suffix_lm_loss=3.8917 | lr=9.92e-05 | train_directions=B_to_A
2026-03-09 15:11:45,385 | INFO | [Step 0100] total_suffix_lm_loss=3.7955 | lr=9.70e-05 | train_directions=B_to_A
2026-03-09 15:12:36,364 | INFO | [Step 0125] total_suffix_lm_loss=3.6894 | lr=9.33e-05 | train_directions=B_to_A
2026-03-09 15:13:27,020 | INFO | [Step 0150] total_suffix_lm_loss=3.5418 | lr=8.83e-05 | train_directions=B_to_A
2026-03-09 15:14:18,377 | INFO | [Step 0175] total_suffix_lm_loss=3.6357 | lr=8.21e-05 | train_directions=B_to_A
2026-03-09 15:15:10,401 |

In [7]:
import logging
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Optional

import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, IterableDataset


@dataclass
class EvalConfig:
    checkpoint_path: str = "./outputs/lsc_toy/final_checkpoint.pt"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    batch_size: int = 1
    num_workers: int = 0
    max_examples_per_dataset: int = 512
    seed: int = 42

    shuffle_eval_stream: bool = True
    shuffle_buffer: int = 10_000

    num_generation_samples_per_dataset: int = 2

    log_filename: str = "eval.log"


CONFIG = EvalConfig()


@dataclass
class HFDatasetSpec:
    name_for_log: str
    dataset_path: str
    dataset_name: Optional[str]
    split: str
    answer_mode: str
    question_field: str = "question"
    streaming: bool = False


class HFQAPairStream(IterableDataset):
    def __init__(
        self,
        spec: HFDatasetSpec,
        max_examples: int,
        shuffle: bool,
        seed: int,
        shuffle_buffer: int,
    ) -> None:
        super().__init__()
        self.spec = spec
        self.max_examples = max_examples
        self.shuffle = shuffle
        self.seed = seed
        self.shuffle_buffer = shuffle_buffer

    def _load_dataset(self):
        candidates = [(self.spec.dataset_path, self.spec.dataset_name)]

        last_error = None
        for dataset_path, dataset_name in candidates:
            try:
                if dataset_name is None:
                    return load_dataset(
                        dataset_path,
                        split=self.spec.split,
                        streaming=self.spec.streaming,
                    )
                return load_dataset(
                    dataset_path,
                    dataset_name,
                    split=self.spec.split,
                    streaming=self.spec.streaming,
                )
            except Exception as exc:
                last_error = exc

        raise RuntimeError(
            f"Failed to load dataset {self.spec.name_for_log} "
            f"with candidates={candidates}"
        ) from last_error

    def __iter__(self):
        dataset = self._load_dataset()
        if self.shuffle:
            if self.spec.streaming:
                dataset = dataset.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)
            else:
                dataset = dataset.shuffle(seed=self.seed)

        emitted = 0

        for example in dataset:
            qa_pair = extract_question_and_answer(self.spec, example)
            if qa_pair is None:
                continue

            yield qa_pair
            emitted += 1
            if emitted >= self.max_examples:
                return


class RunningAverage:
    def __init__(self) -> None:
        self.cosine_sum = 0.0
        self.accuracy_sum = 0.0
        self.native_accuracy_sum = 0.0
        self.count = 0

    def update(self, cosine_value: float, accuracy_value: float, native_accuracy_value: float, n: int) -> None:
        self.cosine_sum += float(cosine_value) * n
        self.accuracy_sum += float(accuracy_value) * n
        self.native_accuracy_sum += float(native_accuracy_value) * n
        self.count += n

    def summary(self) -> Dict[str, float]:
        if self.count == 0:
            return {
                "cosine": float("nan"),
                "accuracy": float("nan"),
                "native_accuracy": float("nan"),
                "count": 0,
            }
        return {
            "cosine": self.cosine_sum / self.count,
            "accuracy": self.accuracy_sum / self.count,
            "native_accuracy": self.native_accuracy_sum / self.count,
            "count": self.count,
        }


def setup_logger(log_path: Path) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("lsc_eval")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger


def build_eval_dataloader(
    spec: HFDatasetSpec,
    eval_config: EvalConfig,
) -> DataLoader:
    dataset = HFQAPairStream(
        spec=spec,
        max_examples=eval_config.max_examples_per_dataset,
        shuffle=eval_config.shuffle_eval_stream,
        seed=eval_config.seed,
        shuffle_buffer=eval_config.shuffle_buffer,
    )
    return DataLoader(
        dataset,
        batch_size=eval_config.batch_size,
        num_workers=eval_config.num_workers,
        collate_fn=lambda batch: batch,
    )


def build_sample_dataloader(
    spec: HFDatasetSpec,
    eval_config: EvalConfig,
) -> DataLoader:
    dataset = HFQAPairStream(
        spec=spec,
        max_examples=eval_config.num_generation_samples_per_dataset,
        shuffle=False,
        seed=eval_config.seed,
        shuffle_buffer=eval_config.shuffle_buffer,
    )
    return DataLoader(
        dataset,
        batch_size=1,
        num_workers=eval_config.num_workers,
        collate_fn=lambda batch: batch,
    )


def extract_question_and_answer(spec: HFDatasetSpec, example: Dict) -> Optional[Dict[str, str]]:
    question = example.get(spec.question_field, "")
    if not isinstance(question, str) or not question.strip():
        return None

    if spec.answer_mode != "boolq":
        raise ValueError(f"Unsupported answer_mode: {spec.answer_mode}")

    answer_value = example.get("answer", None)
    if not isinstance(answer_value, bool):
        return None

    return {
        "question": question.strip(),
        "answer": "yes" if answer_value else "no",
    }


def format_question_prefix(question: str) -> str:
    return f"Question: {question.strip()}\nAnswer:"


def prepare_question_prefix(tokenizer, question: str, device: str) -> Dict[str, torch.Tensor]:
    prefix_text = format_question_prefix(question)
    tokenized = tokenizer(prefix_text, return_tensors="pt")
    input_ids = tokenized.input_ids.to(device)
    if input_ids.shape[1] < 2:
        raise ValueError("Question prefix must tokenize to at least 2 tokens.")
    cache_ids = input_ids[:, :-1]
    seed_token = input_ids[:, -1:]
    return {
        "prefix_text": prefix_text,
        "full_prefix_ids": input_ids,
        "cache_ids": cache_ids,
        "seed_token": seed_token,
    }


def build_boolq_choice_token_ids(tokenizer) -> Dict[str, torch.Tensor]:
    choice_token_ids = {}
    for label in ("yes", "no"):
        token_ids = tokenizer(f" {label}", add_special_tokens=False).input_ids
        if len(token_ids) < 1:
            raise ValueError(f"Failed to tokenize BoolQ label: {label}")
        choice_token_ids[label] = torch.tensor(token_ids, dtype=torch.long)
    return choice_token_ids


def score_candidate_logprob(
    model,
    past_key_values,
    seed_token: torch.Tensor,
    candidate_token_ids: torch.Tensor,
) -> float:
    device = seed_token.device
    candidate_ids = candidate_token_ids.to(device).unsqueeze(0)

    if candidate_ids.shape[1] == 1:
        scoring_input_ids = seed_token
    else:
        scoring_input_ids = torch.cat([seed_token, candidate_ids[:, :-1]], dim=1)

    outputs = model(
        input_ids=scoring_input_ids,
        past_key_values=past_key_values,
        use_cache=False,
    )
    log_probs = F.log_softmax(outputs.logits, dim=-1)
    token_log_probs = log_probs.gather(-1, candidate_ids.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum(dim=1).item()


def score_boolq_choices(
    model,
    past_key_values,
    seed_token: torch.Tensor,
    boolq_choice_token_ids: Dict[str, torch.Tensor],
) -> Dict[str, float]:
    return {
        "yes": score_candidate_logprob(
            model=model,
            past_key_values=past_key_values,
            seed_token=seed_token,
            candidate_token_ids=boolq_choice_token_ids["yes"],
        ),
        "no": score_candidate_logprob(
            model=model,
            past_key_values=past_key_values,
            seed_token=seed_token,
            candidate_token_ids=boolq_choice_token_ids["no"],
        ),
    }


def predict_boolq_label(choice_scores: Dict[str, float]) -> str:
    return "yes" if choice_scores["yes"] >= choice_scores["no"] else "no"


def summarize_path_metrics(path_metrics: Dict[str, RunningAverage]) -> Dict[str, Dict[str, float]]:
    results = {}
    total_cosine_sum = 0.0
    total_accuracy_sum = 0.0
    total_native_accuracy_sum = 0.0
    total_count = 0

    for path_name, meter in path_metrics.items():
        path_result = meter.summary()
        results[path_name] = path_result

        total_cosine_sum += meter.cosine_sum
        total_accuracy_sum += meter.accuracy_sum
        total_native_accuracy_sum += meter.native_accuracy_sum
        total_count += meter.count

    if total_count == 0:
        results["AVG"] = {
            "cosine": float("nan"),
            "accuracy": float("nan"),
            "native_accuracy": float("nan"),
            "count": 0,
        }
    else:
        results["AVG"] = {
            "cosine": total_cosine_sum / total_count,
            "accuracy": total_accuracy_sum / total_count,
            "native_accuracy": total_native_accuracy_sum / total_count,
            "count": total_count,
        }

    return results


def summarize_overall_results(all_results: Dict[str, Dict[str, Dict[str, float]]], active_directions) -> Dict[str, Dict[str, float]]:
    overall = {
        direction: {"cosine_sum": 0.0, "accuracy_sum": 0.0, "native_accuracy_sum": 0.0, "count": 0}
        for direction in active_directions
    }

    for dataset_result in all_results.values():
        for direction in active_directions:
            row = dataset_result[direction]
            count = int(row["count"])
            overall[direction]["cosine_sum"] += float(row["cosine"]) * count
            overall[direction]["accuracy_sum"] += float(row["accuracy"]) * count
            overall[direction]["native_accuracy_sum"] += float(row["native_accuracy"]) * count
            overall[direction]["count"] += count

    summarized = {}
    total_cosine_sum = 0.0
    total_accuracy_sum = 0.0
    total_native_accuracy_sum = 0.0
    total_count = 0

    for direction in active_directions:
        count = overall[direction]["count"]
        if count == 0:
            summarized[direction] = {
                "cosine": float("nan"),
                "accuracy": float("nan"),
                "native_accuracy": float("nan"),
                "count": 0,
            }
        else:
            summarized[direction] = {
                "cosine": overall[direction]["cosine_sum"] / count,
                "accuracy": overall[direction]["accuracy_sum"] / count,
                "native_accuracy": overall[direction]["native_accuracy_sum"] / count,
                "count": count,
            }

        total_cosine_sum += overall[direction]["cosine_sum"]
        total_accuracy_sum += overall[direction]["accuracy_sum"]
        total_native_accuracy_sum += overall[direction]["native_accuracy_sum"]
        total_count += count

    if total_count == 0:
        summarized["AVG"] = {
            "cosine": float("nan"),
            "accuracy": float("nan"),
            "native_accuracy": float("nan"),
            "count": 0,
        }
    else:
        summarized["AVG"] = {
            "cosine": total_cosine_sum / total_count,
            "accuracy": total_accuracy_sum / total_count,
            "native_accuracy": total_native_accuracy_sum / total_count,
            "count": total_count,
        }

    return summarized


def build_direction_pretty_name(direction: str, model_a_id: str, model_b_id: str) -> str:
    if direction == "A_to_B":
        return f"A_to_B ({model_a_id} -> {model_b_id})"
    if direction == "B_to_A":
        return f"B_to_A ({model_b_id} -> {model_a_id})"
    return direction


@torch.inference_mode()
def evaluate_dataset(
    spec: HFDatasetSpec,
    dataloader: DataLoader,
    tokenizer,
    train_config,
    translator_pool,
    model_specs,
    model_a,
    model_b,
    active_directions,
    logger: logging.Logger,
) -> Dict[str, Dict[str, float]]:
    device = train_config.device
    boolq_choice_token_ids = build_boolq_choice_token_ids(tokenizer)

    path_metrics = {
        direction: RunningAverage()
        for direction in active_directions
    }

    processed_examples = 0

    for batch_idx, batch in enumerate(dataloader, start=1):
        for example in batch:
            question = example["question"]
            gold_answer = example["answer"]

            prefix = prepare_question_prefix(tokenizer=tokenizer, question=question, device=device)
            prefix_cache_ids = prefix["cache_ids"]
            seed_token = prefix["seed_token"]

            past_a = extract_past_key_values(model_a, prefix_cache_ids)
            past_b = extract_past_key_values(model_b, prefix_cache_ids)

            direction_contexts = {
                "A_to_B": {
                    "source_past": past_a,
                    "source_name": "A",
                    "target_name": "B",
                    "target_spec": model_specs["B"],
                    "target_full_past": past_b,
                    "target_model": model_b,
                },
                "B_to_A": {
                    "source_past": past_b,
                    "source_name": "B",
                    "target_name": "A",
                    "target_spec": model_specs["A"],
                    "target_full_past": past_a,
                    "target_model": model_a,
                },
            }

            for direction in active_directions:
                context = direction_contexts[direction]

                translated_top_past = translator_pool.translate_top_layers(
                    past_key_values=context["source_past"],
                    src_name=context["source_name"],
                    dst_name=context["target_name"],
                    dst_spec=context["target_spec"],
                )

                target_top = slice_top_layers(
                    past_key_values=context["target_full_past"],
                    top_layers_to_translate=train_config.top_layers_to_translate,
                )
                cosine_value = cosine_similarity_between_past(translated_top_past, target_top)

                mixed_target_past = replace_top_layers(
                    base_past_key_values=context["target_full_past"],
                    translated_top_past_key_values=translated_top_past,
                )

                translated_scores = score_boolq_choices(
                    model=context["target_model"],
                    past_key_values=mixed_target_past,
                    seed_token=seed_token,
                    boolq_choice_token_ids=boolq_choice_token_ids,
                )
                native_scores = score_boolq_choices(
                    model=context["target_model"],
                    past_key_values=context["target_full_past"],
                    seed_token=seed_token,
                    boolq_choice_token_ids=boolq_choice_token_ids,
                )

                translated_pred = predict_boolq_label(translated_scores)
                native_pred = predict_boolq_label(native_scores)

                acc = 1.0 if translated_pred == gold_answer else 0.0
                native_acc = 1.0 if native_pred == gold_answer else 0.0

                path_metrics[direction].update(cosine_value, acc, native_acc, 1)

            processed_examples += 1

        if batch_idx % 50 == 0:
            logger.info(
                "[%s] progress: %d/%d examples",
                spec.name_for_log,
                processed_examples,
                CONFIG.max_examples_per_dataset,
            )

    return summarize_path_metrics(path_metrics)


def log_dataset_result(
    logger: logging.Logger,
    dataset_name: str,
    results: Dict[str, Dict[str, float]],
    model_a_id: str,
    model_b_id: str,
    active_directions,
) -> None:
    logger.info("===== %s =====", dataset_name)
    for direction in list(active_directions) + ["AVG"]:
        row = results[direction]
        pretty_name = (
            "AVG (weighted over evaluated paths)"
            if direction == "AVG"
            else build_direction_pretty_name(direction, model_a_id, model_b_id)
        )
        logger.info(
            "%s | cosine=%.6f | accuracy=%.6f | native_accuracy=%.6f | count=%d",
            pretty_name,
            row["cosine"],
            row["accuracy"],
            row["native_accuracy"],
            int(row["count"]),
        )


def log_overall_result(
    logger: logging.Logger,
    results: Dict[str, Dict[str, float]],
    model_a_id: str,
    model_b_id: str,
    active_directions,
) -> None:
    logger.info("===== OVERALL AVERAGE ACROSS DATASETS =====")
    for direction in list(active_directions) + ["AVG"]:
        row = results[direction]
        pretty_name = (
            "AVG (weighted over all datasets and evaluated paths)"
            if direction == "AVG"
            else build_direction_pretty_name(direction, model_a_id, model_b_id)
        )
        logger.info(
            "%s | cosine=%.6f | accuracy=%.6f | native_accuracy=%.6f | count=%d",
            pretty_name,
            row["cosine"],
            row["accuracy"],
            row["native_accuracy"],
            int(row["count"]),
        )


@torch.inference_mode()
def log_boolq_score_samples(
    spec: HFDatasetSpec,
    tokenizer,
    train_config,
    eval_config: EvalConfig,
    translator_pool,
    model_specs,
    model_a,
    model_b,
    active_directions,
    logger: logging.Logger,
) -> None:
    if eval_config.num_generation_samples_per_dataset <= 0:
        return

    sample_dataloader = build_sample_dataloader(
        spec=spec,
        eval_config=eval_config,
    )
    boolq_choice_token_ids = build_boolq_choice_token_ids(tokenizer)

    logger.info("===== SAMPLE BOOLQ SCORES: %s =====", spec.name_for_log)

    sample_idx = 0
    for batch in sample_dataloader:
        example = batch[0]
        question = example["question"]
        gold_answer = example["answer"]

        prefix = prepare_question_prefix(tokenizer=tokenizer, question=question, device=train_config.device)
        prefix_cache_ids = prefix["cache_ids"]
        seed_token = prefix["seed_token"]
        prefix_text = prefix["prefix_text"]

        past_a = extract_past_key_values(model_a, prefix_cache_ids)
        past_b = extract_past_key_values(model_b, prefix_cache_ids)

        direction_contexts = {
            "A_to_B": {
                "source_past": past_a,
                "source_name": "A",
                "target_name": "B",
                "target_spec": model_specs["B"],
                "target_full_past": past_b,
                "target_model": model_b,
            },
            "B_to_A": {
                "source_past": past_b,
                "source_name": "B",
                "target_name": "A",
                "target_spec": model_specs["A"],
                "target_full_past": past_a,
                "target_model": model_a,
            },
        }

        logger.info(
            "--- Sample %d/%d | %s ---",
            sample_idx + 1,
            eval_config.num_generation_samples_per_dataset,
            spec.name_for_log,
        )
        logger.info("prefix(question):\n%s", prefix_text)
        logger.info("gold_answer: %s", gold_answer)

        for direction in active_directions:
            context = direction_contexts[direction]

            translated_top_past = translator_pool.translate_top_layers(
                past_key_values=context["source_past"],
                src_name=context["source_name"],
                dst_name=context["target_name"],
                dst_spec=context["target_spec"],
            )
            mixed_target_past = replace_top_layers(
                base_past_key_values=context["target_full_past"],
                translated_top_past_key_values=translated_top_past,
            )

            translated_scores = score_boolq_choices(
                model=context["target_model"],
                past_key_values=mixed_target_past,
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )
            native_scores = score_boolq_choices(
                model=context["target_model"],
                past_key_values=context["target_full_past"],
                seed_token=seed_token,
                boolq_choice_token_ids=boolq_choice_token_ids,
            )

            translated_pred = predict_boolq_label(translated_scores)
            native_pred = predict_boolq_label(native_scores)
            pretty_name = build_direction_pretty_name(direction, train_config.model_a_id, train_config.model_b_id)

            logger.info(
                "%s translated_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
                pretty_name,
                translated_scores["yes"],
                translated_scores["no"],
                translated_pred,
                translated_pred == gold_answer,
            )
            logger.info(
                "%s native_baseline_scores: yes=%.6f | no=%.6f | pred=%s | correct=%s",
                pretty_name,
                native_scores["yes"],
                native_scores["no"],
                native_pred,
                native_pred == gold_answer,
            )

        sample_idx += 1
        if sample_idx >= eval_config.num_generation_samples_per_dataset:
            break


def main() -> None:
    eval_config = CONFIG
    set_seed(eval_config.seed)

    checkpoint_path = Path(eval_config.checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    log_path = checkpoint_path.parent / eval_config.log_filename
    logger = setup_logger(log_path)
    logger.info("Starting evaluation")
    logger.info("checkpoint_path=%s", checkpoint_path)
    logger.info("eval_config=%s", asdict(eval_config))

    (
        train_config,
        translator_pool,
        model_specs,
        model_a,
        model_b,
        tokenizer,
    ) = load_translator_pool_from_checkpoint(
        checkpoint_path=str(checkpoint_path),
        device_override=eval_config.device,
    )

    active_directions = parse_train_directions(train_config.train_directions)

    translator_pool.eval()
    model_a.eval()
    model_b.eval()

    logger.info("restored_train_config=%s", asdict(train_config))
    logger.info("model_A=%s", train_config.model_a_id)
    logger.info("model_B=%s", train_config.model_b_id)
    logger.info("top_layers_to_translate=%d", train_config.top_layers_to_translate)
    logger.info("active_directions=%s", active_directions)
    logger.info("translation_mode=replace_top_layers_after_target_forward")
    logger.info("qa_eval_log_path=%s", log_path)

    dataset_specs = [
        HFDatasetSpec(
            name_for_log="BoolQ/validation",
            dataset_path="boolq",
            dataset_name=None,
            split="validation",
            answer_mode="boolq",
            question_field="question",
            streaming=False,
        ),
    ]

    all_results = {}

    for spec in dataset_specs:
        logger.info("Preparing dataloader for %s", spec.name_for_log)
        dataloader = build_eval_dataloader(
            spec=spec,
            eval_config=eval_config,
        )

        results = evaluate_dataset(
            spec=spec,
            dataloader=dataloader,
            tokenizer=tokenizer,
            train_config=train_config,
            translator_pool=translator_pool,
            model_specs=model_specs,
            model_a=model_a,
            model_b=model_b,
            active_directions=active_directions,
            logger=logger,
        )
        all_results[spec.name_for_log] = results

        log_dataset_result(
            logger=logger,
            dataset_name=spec.name_for_log,
            results=results,
            model_a_id=train_config.model_a_id,
            model_b_id=train_config.model_b_id,
            active_directions=active_directions,
        )

        log_boolq_score_samples(
            spec=spec,
            tokenizer=tokenizer,
            train_config=train_config,
            eval_config=eval_config,
            translator_pool=translator_pool,
            model_specs=model_specs,
            model_a=model_a,
            model_b=model_b,
            active_directions=active_directions,
            logger=logger,
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    logger.info("===== FINAL SUMMARY =====")
    for dataset_name, result in all_results.items():
        summary_parts = []
        for direction in active_directions:
            row = result[direction]
            summary_parts.append(
                f"{direction}(cos={row['cosine']:.6f}, acc={row['accuracy']:.6f}, native_acc={row['native_accuracy']:.6f})"
            )
        avg_row = result["AVG"]
        summary_parts.append(
            f"AVG(cos={avg_row['cosine']:.6f}, acc={avg_row['accuracy']:.6f}, native_acc={avg_row['native_accuracy']:.6f})"
        )
        logger.info("%s | %s", dataset_name, " | ".join(summary_parts))

    overall_results = summarize_overall_results(all_results, active_directions)
    log_overall_result(
        logger=logger,
        results=overall_results,
        model_a_id=train_config.model_a_id,
        model_b_id=train_config.model_b_id,
        active_directions=active_directions,
    )

    logger.info("Done. Saved log to %s", log_path)


if __name__ == "__main__":
    main()

2026-03-09 15:25:36,961 | INFO | Starting evaluation
2026-03-09 15:25:36,962 | INFO | checkpoint_path=outputs/lsc_toy/final_checkpoint.pt
2026-03-09 15:25:36,963 | INFO | eval_config={'checkpoint_path': './outputs/lsc_toy/final_checkpoint.pt', 'device': 'cuda', 'batch_size': 1, 'num_workers': 0, 'max_examples_per_dataset': 512, 'seed': 42, 'shuffle_eval_stream': True, 'shuffle_buffer': 10000, 'num_generation_samples_per_dataset': 2, 'log_filename': 'eval.log'}
2026-03-09 15:25:47,808 | INFO | restored_train_config={'model_a_id': 'gpt2', 'model_b_id': 'gpt2-medium', 'output_dir': './outputs/lsc_toy', 'max_steps': 500, 'batch_size': 1, 'grad_accum_steps': 16, 'total_tokens': 128, 'prefix_tokens': 64, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'warmup_steps': 50, 'grad_clip_norm': 1.0, 'log_every': 25, 'save_every': 100, 'seed': 42, 'shuffle_buffer': 50000, 'top_layers_to_translate': 6, 'translator_dim': 1024, 'translator_heads': 16, 'translator_depth': 2, 'translator_mlp_ratio': 2, '

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.69M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9427 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3270 [00:00<?, ? examples/s]

2026-03-09 15:25:55,828 | INFO | [BoolQ/validation] progress: 50/512 examples
2026-03-09 15:26:00,580 | INFO | [BoolQ/validation] progress: 100/512 examples
2026-03-09 15:26:05,302 | INFO | [BoolQ/validation] progress: 150/512 examples
2026-03-09 15:26:10,033 | INFO | [BoolQ/validation] progress: 200/512 examples
2026-03-09 15:26:14,711 | INFO | [BoolQ/validation] progress: 250/512 examples
2026-03-09 15:26:19,359 | INFO | [BoolQ/validation] progress: 300/512 examples
2026-03-09 15:26:24,043 | INFO | [BoolQ/validation] progress: 350/512 examples
2026-03-09 15:26:28,749 | INFO | [BoolQ/validation] progress: 400/512 examples
2026-03-09 15:26:33,509 | INFO | [BoolQ/validation] progress: 450/512 examples
2026-03-09 15:26:38,319 | INFO | [BoolQ/validation] progress: 500/512 examples
2026-03-09 15:26:39,458 | INFO | ===== BoolQ/validation =====
2026-03-09 15:26:39,459 | INFO | B_to_A (gpt2-medium -> gpt2) | cosine=0.157284 | accuracy=0.548828 | native_accuracy=0.550781 | count=512
2026-03-09